# Pré-processamento 5 — Triagem de concatenações

In [7]:
from pathlib import Path
import pandas as pd
from functools import lru_cache

arquivo = "Corpus_SANTOS_TONI_FDC_tratado_sem_repeticoes.xlsx"
base = pd.read_excel(arquivo)
palavras = set(base["PALAVRA"].dropna().str.casefold())
pistas = {"não", "está", "você", "também", "então", "aqui", "agora",
          "meu", "minha", "para", "isso", "quero", "vamos", "tenho",
          "pode", "tem", "com", "sem", "uma", "ele", "ela"}

def separar(palavra):
    @lru_cache(None)
    def partes(inicio):
        if inicio == len(palavra):
            return [()]
        resultado = []
        for fim in range(inicio + 2, len(palavra) + 1):
            trecho = palavra[inicio:fim]
            if trecho in palavras and trecho != palavra:
                for resto in partes(fim):
                    if len(resto) < 3:
                        resultado.append((trecho, *resto))
        return resultado

    opcoes = partes(0)
    if opcoes:
        return min(opcoes, key=lambda p: (-sum(t in pistas for t in p), len(p), -min(map(len, p)), p))
    return ()

linhas = []
for palavra, lema, freq in base[["PALAVRA", "LEMA", "FREQ"]].itertuples(index=False, name=None):
    if not isinstance(palavra, str) or not palavra.isalpha() or len(palavra) < 5:
        continue
    partes = separar(palavra.casefold())
    fortes = [p for p in partes if p in pistas]
    repeticao = len(partes) > 1 and len(set(partes)) == 1
    if repeticao:
        prioridade, motivo = 0, "Repetição da mesma palavra"
    elif fortes:
        prioridade, motivo = 1, "Separação inclui " + ", ".join(fortes)
    elif partes and len(palavra) >= 8:
        prioridade, motivo = 2, "Separável em palavras da base"
    elif len(palavra) >= 12:
        prioridade, motivo = 3, "Comprimento de 12 letras ou mais"
    else:
        continue
    linhas.append((prioridade, palavra, lema, freq, " + ".join(partes) or "—", motivo))

triagem = pd.DataFrame(linhas, columns=["prioridade", "PALAVRA", "LEMA", "FREQ", "Separação", "Motivo"])
triagem = triagem.sort_values(["prioridade", "FREQ", "PALAVRA"], ascending=[True, False, True])
triagem = triagem.drop(columns="prioridade").reset_index(drop=True)
triagem.index += 1
print("Candidatos encontrados:", len(triagem))
saida = Path("Lista_triagem_concatenacoes.html")
tabela = triagem.rename_axis("Nº").to_html()
saida.write_text(
    '<!doctype html><html lang="pt-BR"><meta charset="utf-8">'
    '<title>Triagem de concatenações</title>'
    '<h1>Triagem de concatenações</h1>'
    f'<p>{len(triagem)} candidatos para revisão manual.</p>'
    + tabela + '</html>',
    encoding="utf-8",
)
print("Lista completa gerada:", saida.resolve())
display(triagem.head(20))

Candidatos encontrados: 525
Lista completa gerada: C:\Users\larin\Downloads\PRÉ PROCESSAMENTO V2.0\Lista_triagem_concatenacoes.html


,PALAVRA,LEMA,FREQ,Separação,Motivo
1,bombom,bombom,9,bom + bom,Repetição da mesma palavra
2,pulapula,pulapula,8,pula + pula,Repetição da mesma palavra
3,ticotico,ticotico,6,tico + tico,Repetição da mesma palavra
4,piuípiuípiuí,piuípiuípiuí,1,piuí + piuí + piuí,Repetição da mesma palavra
5,senão,senão,222,se + não,Separação inclui não
6,comeu,comer,200,com + eu,Separação inclui com
7,amarela,amanhã,57,amar + ela,Separação inclui ela
8,parado,parar,13,para + do,Separação inclui para
9,separa,separar,10,se + para,Separação inclui para
10,comemorando,comemorar,8,com + em + orando,Separação inclui com


In [5]:
# Exclusões aprovadas na revisão até a posição 200.
remover = {
    "xixixixi",
    "lambelambe",
    "tititi",
    "aiaiaiai",
    "batebate",
    "chuchuchuchu",
    "cocococo",
    "lalalalalalalalala",
    "agoraagora",
    "blindonblindon",
    "bubububu",
    "cliqueclique",
    "eieiei",
    "hahahahahaha",
    "lalalalalalalalalala",
    "maumau",
    "maçamaçamaça",
    "nananana",
    "nononono",
    "olhaolhaolha",
    "pegapega",
    "piupiupiupiu",
    "qualqualqual",
    "rodaroda",
    "tatatata",
    "temtem",
    "tiquetique",
    "tiquetiquetiquetique",
    "tudotudo",
    "aquicom",
    "aquideixa",
    "cabeçapode",
    "dexoveaqui",
    "essaaqui",
    "marcarcom",
    "olhaaqui",
    "parade",
    "paraeu",
    "paraonde",
    "paravoar",
    "vamosbrincar",
    "vamosfazer",
    "corderosa",
    "cocacola",
    "luluzona",
    "aiaiaiaiai",
    "brancadeneve",
    "grandepeixe",
    "pequenasereia",
    "piupiupiupiupiupiupiu",
    "tumtumtumtumtum",
    "peladona",
    "supernuvem",
    "muitobem",
    "babirussa",
    "caiofelipe",
    "casteloratimbum",
    "curative",
    "harryporter",
    "harrypotter",
    "papapapapapapapapapapa",
    "piupiupiupiupiu",
    "pontacabeça",
    "putzgrila",
}


In [6]:
# Execute depois da célula que define a lista remover.
import pandas as pd

arquivo = "Corpus_SANTOS_TONI_FDC_tratado_sem_repeticoes.xlsx"
base_atual = pd.read_excel(arquivo)
mascara_remover = base_atual["PALAVRA"].isin(remover)
encontradas = base_atual.loc[mascara_remover, "PALAVRA"].value_counts()
resultado = base_atual.loc[~mascara_remover].copy()

print("=== PALAVRAS QUE SERÃO REMOVIDAS ===")
for palavra, quantidade in encontradas.items():
    print(f"{palavra}: {quantidade}")
print("Registros que serão removidos:", int(mascara_remover.sum()))

if mascara_remover.any():
    resultado.to_excel(arquivo, index=False)
    print("Arquivo atualizado:", arquivo)
else:
    print("Nenhuma exclusão pendente nesta lista.")

print("Total de registros restantes:", len(resultado))
print("Repetições restantes em PALAVRA:", int(resultado["PALAVRA"].duplicated().sum()))


=== PALAVRAS QUE SERÃO REMOVIDAS ===
agoraagora: 1
aiaiaiai: 1
aiaiaiaiai: 1
aquicom: 1
aquideixa: 1
babirussa: 1
batebate: 1
blindonblindon: 1
brancadeneve: 1
bubububu: 1
cabeçapode: 1
caiofelipe: 1
casteloratimbum: 1
chuchuchuchu: 1
cliqueclique: 1
cocacola: 1
cocococo: 1
corderosa: 1
curative: 1
dexoveaqui: 1
eieiei: 1
essaaqui: 1
grandepeixe: 1
hahahahahaha: 1
harryporter: 1
harrypotter: 1
lalalalalalalalala: 1
lalalalalalalalalala: 1
lambelambe: 1
luluzona: 1
marcarcom: 1
maumau: 1
maçamaçamaça: 1
muitobem: 1
nananana: 1
nononono: 1
olhaaqui: 1
olhaolhaolha: 1
parade: 1
paraeu: 1
paraonde: 1
paravoar: 1
pegapega: 1
peladona: 1
pequenasereia: 1
piupiupiupiu: 1
piupiupiupiupiu: 1
piupiupiupiupiupiupiu: 1
pontacabeça: 1
putzgrila: 1
qualqualqual: 1
rodaroda: 1
supernuvem: 1
tatatata: 1
temtem: 1
tiquetique: 1
tiquetiquetiquetique: 1
tititi: 1
tudotudo: 1
tumtumtumtumtum: 1
vamosbrincar: 1
vamosfazer: 1
xixixixi: 1
Registros que serão removidos: 63
Arquivo atualizado: Corpus_SANTOS_TO